In [10]:
raw_path = os.path.join(RAW_DIR, "synthetic_demand_raw.csv")
raw_df.to_csv(raw_path, index=False)

processed_df = raw_df.copy()
processed_df[demand_cols] = processed_df[demand_cols].astype(int)
processed_df["data_source"] = "synthetic"     # clear label
processed_path = os.path.join(PROCESSED_DIR, "synthetic_demand_clean.csv")
processed_df.to_csv(processed_path, index=False)

desc_table.to_csv(os.path.join(TABLES_DIR, "demand_descriptive_statistics.csv"), index=False)

# Reload to confirm the files are readable and unchanged
check_df = pd.read_csv(processed_path)
print(check_df.shape)
print(check_df.head())


(10000, 5)
   scenario_id  poisson  negbin_var240  negbin_var480 data_source
0            1      129            118            171   synthetic
1            2      134            139            126   synthetic
2            3      129            113            117   synthetic
3            4      105             93            124   synthetic
4            5      116             94            132   synthetic


## Module 06 summary
- Generated 10,000 synthetic demand scenarios (seed = 42) under three distributions, all with target mean 120: Poisson (variance 120), Negative Binomial (variance 240), and Negative Binomial (variance 480).
- This data is **synthetic**, not real company data.
- Validation: all demand values are non-negative integers, there are no missing values, and sample means are within sampling error of 120.
- Sample variances (118.3, 242.4, 464.8) are close to the targets (120, 240, 480). Small gaps are due to random sampling.
- Maximum observed demand (213) is far below Qmax = 500.
- This dataset will serve as the **training** scenarios. Independent test scenarios (different seed) will be generated in Module 10.

In [9]:
rows = []
for col in demand_cols:
    s = raw_df[col]
    rows.append({
        "distribution": col,
        "target_mean": MEAN_DEMAND,
        "target_variance": target_vars[col],
        "sample_mean": s.mean(),
        "sample_variance": s.var(ddof=1),
        "std_dev": s.std(ddof=1),
        "min": s.min(),
        "p5": s.quantile(0.05),
        "median": s.median(),
        "p95": s.quantile(0.95),
        "max": s.max(),
    })

desc_table = pd.DataFrame(rows).round(2)
desc_table

,distribution,target_mean,target_variance,sample_mean,sample_variance,std_dev,min,p5,median,p95,max
0,poisson,120,120,120.02,118.34,10.88,80,103.0,120.0,138.0,160
1,negbin_var240,120,240,120.02,242.36,15.57,70,95.0,119.0,146.0,187
2,negbin_var480,120,480,119.73,464.84,21.56,49,87.0,118.0,158.0,213


In [8]:
demand_cols = ["poisson"] + list(NB_VARIANCES.keys())
target_vars = {"poisson": MEAN_DEMAND, **NB_VARIANCES}

checks = []
checks.append(("Row count is 10,000", len(raw_df) == N_SCENARIOS))
checks.append(("scenario_id unique", raw_df["scenario_id"].is_unique))
checks.append(("No missing values", not raw_df.isna().any().any()))

for col in demand_cols:
    checks.append((f"{col}: integer dtype", pd.api.types.is_integer_dtype(raw_df[col])))
    checks.append((f"{col}: all demand >= 0", bool((raw_df[col] >= 0).all())))
    # sample mean should be within 4 standard errors of the target mean
    std_err = np.sqrt(target_vars[col] / N_SCENARIOS)
    checks.append((f"{col}: sample mean within 4 SE of {MEAN_DEMAND}",
                   abs(raw_df[col].mean() - MEAN_DEMAND) < 4 * std_err))

for description, passed in checks:
    print(("PASS  " if passed else "FAIL  ") + description)

assert all(passed for _, passed in checks), "Validation failed - do not save processed data."

PASS  Row count is 10,000
PASS  scenario_id unique
PASS  No missing values
PASS  poisson: integer dtype
PASS  poisson: all demand >= 0
PASS  poisson: sample mean within 4 SE of 120
PASS  negbin_var240: integer dtype
PASS  negbin_var240: all demand >= 0
PASS  negbin_var240: sample mean within 4 SE of 120
PASS  negbin_var480: integer dtype
PASS  negbin_var480: all demand >= 0
PASS  negbin_var480: sample mean within 4 SE of 120


In [7]:
rng = np.random.default_rng(SEED)

raw_df = pd.DataFrame({"scenario_id": np.arange(1, N_SCENARIOS + 1)})
raw_df["poisson"] = rng.poisson(lam=MEAN_DEMAND, size=N_SCENARIOS)

for name, var in NB_VARIANCES.items():
    n, p = negbin_params(MEAN_DEMAND, var)
    raw_df[name] = rng.negative_binomial(n, p, size=N_SCENARIOS)

raw_df.head()

,scenario_id,poisson,negbin_var240,negbin_var480
0,1,129,118,171
1,2,134,139,126
2,3,129,113,117
3,4,105,93,124
4,5,116,94,132


In [6]:
def negbin_params(mean, variance):
    """Convert a target mean and variance into NumPy's (n, p) for negative binomial."""
    if variance <= mean:
        raise ValueError("Negative binomial needs variance > mean.")
    p = mean / variance
    n = mean**2 / (variance - mean)
    return n, p

# Verify the theory BEFORE generating any data
for name, var in NB_VARIANCES.items():
    n, p = negbin_params(MEAN_DEMAND, var)
    theo_mean, theo_var = stats.nbinom.stats(n, p, moments="mv")
    print(f"{name}: n={n:.2f}, p={p:.3f} -> theoretical mean={float(theo_mean):.2f}, variance={float(theo_var):.2f}")

negbin_var240: n=120.00, p=0.500 -> theoretical mean=120.00, variance=240.00
negbin_var480: n=40.00, p=0.250 -> theoretical mean=120.00, variance=480.00


In [2]:
import os

for path in [os.path.join("..", "data", "raw"),
             os.path.join("..", "data", "processed"),
             os.path.join("..", "results", "tables")]:
    print(path,
          "| exists:", os.path.exists(path),
          "| is folder:", os.path.isdir(path),
          "| is file:", os.path.isfile(path))

..\data\raw | exists: True | is folder: False | is file: True
..\data\processed | exists: True | is folder: False | is file: True
..\results\tables | exists: False | is folder: False | is file: False


In [5]:
import os
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42                 # fixed seed for reproducibility
N_SCENARIOS = 10_000      # number of demand scenarios
MEAN_DEMAND = 120         # target mean for all distributions

# Target variances for the negative binomial versions (Poisson variance = 120)
NB_VARIANCES = {"negbin_var240": 240, "negbin_var480": 480}

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
TABLES_DIR = os.path.join("..", "results", "tables")
for folder in (RAW_DIR, PROCESSED_DIR, TABLES_DIR):
    os.makedirs(folder, exist_ok=True)
    